### Model Serving

The dataset used for this example is obtained from [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing).

This part consists of serving our machine learning model. We are going to study online inference and batch inference.

#### Machine Learning Model Training

As seen already in Session 2, we are going to train our ML model and track it with MLFlow (we can reuse the code from that session).

In [1]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score
import mlflow
from mlflow.models import infer_signature

In a new terminal, run a local Tracking Server with the following command:

`mlflow server --host 127.0.0.1 --port 8080`

In [2]:
# read dataset
df = pd.read_csv("datasets/bank-full_train_test.csv")

# balance dataset
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Separate the classes
    df_y0 = df[df['y'] == 'no']
    df_y1 = df[df['y'] == 'yes']

    # Find the smaller class size
    min_size = len(df_y1)

    # Randomly sample from each class
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    # Concatenate back together
    df_balanced = pd.concat([df_y0_balanced, df_y1])

    # Shuffle the dataset
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

# map target
df.loc[df["y"] == "yes", "y"] = 1
df.loc[df["y"] == "no", "y"] = 0
df["y"] = df["y"].astype(int)

# prepare dataset for classification model
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = []
        self.BINARY_FEATURES = [
            "housing",
            "loan",
            "default",
        ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(self.DROP_COLUMNS, axis=1)
        df = self._map_binary_column_to_int(df)
        df = self._map_month_to_int(df)

        return df

    def _map_binary_column_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in self.BINARY_FEATURES:
            df[col] = df[col].map({"yes": 1, "no": 0})
        return df

    def _map_month_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        month_mapping = {
            "jan": 1,
            "feb": 2,
            "mar": 3,
            "apr": 4,
            "may": 5,
            "jun": 6,
            "jul": 7,
            "aug": 8,
            "sep": 9,
            "oct": 10,
            "nov": 11,
            "dec": 12,
        }
        df["month"] = df["month"].map(month_mapping)

        return df

# transform the dataset
df = Transformer().transform(df)

# encode categorical variables
ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df[ONE_HOT_ENCODE_COLUMNS])
encoded_df = encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
df = pd.concat([df, encoded_df], axis=1)

# Split the data into training and test sets
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

# Calculate metrics
f1_score = f1_score(y_test, y_pred)

c:\Users\Robert\Documents\EADA\2026_MLOPS_SD\mlops-and-system-design\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Robert\Documents\EADA\2026_MLOPS_SD\mlops-and-system-design\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [3]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("LR Experiment - Serving")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1_score)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model to present MLFlow capabilities")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="bank-model-basic",  # used to register our model
    )

2026/05/19 12:17:15 INFO mlflow.tracking.fluent: Experiment with name 'LR Experiment - Serving' does not exist. Creating a new experiment.
c:\Users\Robert\Documents\EADA\2026_MLOPS_SD\mlops-and-system-design\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'bank-model-basic' already exists. Crea

🏃 View run intrigued-skunk-925 at: http://127.0.0.1:8080/#/experiments/775267522575094143/runs/94318391fcbd491abd49234c43d4f15a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/775267522575094143


Created version '2' of model 'bank-model-basic'.


For part of the exercice, it is necessary to save the encoder and the model. These could be stored in a shared repository, but for the purpose of this exercice, we are going to store it in the  `model_utils/` folder.

In [4]:
import joblib

PATH = 'model_utils/'
joblib.dump(lr, f'{PATH}lr_model.pkl')
joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')

['model_utils/one_hot_encoder.pkl']

#### Online Inference

According to [Google](https://developers.google.com/machine-learning/crash-course/production-ml-systems/static-vs-dynamic-inference), **online inference** means that the model only makes predictions on demand, for example, when a client requests a prediction.

With MLFlow, we can deplooy our model to a local server easily using the mlflow models serve command, which needs to be executed on a new terminal.
`mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`

Note that is necessary to specify the model uri (`runs:/<run_id>/model`) and the port (`-p 5000`).

Before executing the command, we need to specify the tracking uri in the terminal like this:

`export MLFLOW_TRACKING_URI=http://127.0.0.1:8080`

In [6]:
model_uri='runs:/94318391fcbd491abd49234c43d4f15a/bank_model'
# mlflow models serve -m runs:/94318391fcbd491abd49234c43d4f15a/bank_model -p 5000 --no-conda

If you encounter any issues with the server deployment, force your virtual environment to use a version of Starlette prior to the breaking change. Run this in your terminal with your .venv activated:

`pip install "starlette<1.0.0"`

In [5]:
import requests
import json

The predictions can be done through the `/invocations` endpoint, which accepts CSV or JSON inputs. The input format must be specified in the Content-Type header as either application/json or application/csv.

In [7]:
class MLflowInferenceClient:
    ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]
    def __init__(self, host="http://127.0.0.1", port=5000):
        self.url = f"{host}:{port}/invocations"
        self.encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
        self.transformer = Transformer()

    def _apply_transformation(self, df: pd.DataFrame):
        df = self.transformer.transform(df.copy())
        encoded_df = self.encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
        df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
        df = pd.concat([df, encoded_df], axis=1)

        return df 

    def predict_json(self, json_data: dict):
        """Send JSON (pandas-style) request"""
        headers = {"Content-Type": "application/json"}
        response = requests.post(self.url, headers=headers, json=json_data)
        return response

    def predict_csv(self, csv_data: str):
        """Send CSV-formatted request"""
        headers = {"Content-Type": "text/csv"}
        response = requests.post(self.url, headers=headers, data=csv_data)
        return response
    
    def predict_pandas(self, pandas_data: pd.DataFrame):
        headers = {"Content-Type": "application/json"}
        pandas_data = self._apply_transformation(pandas_data)
        payload = {"dataframe_split": pandas_data.to_dict(orient="split")}
        response = requests.post(self.url, headers=headers, data=json.dumps(payload))
        return response
    
ml_flow_online_inference_client = MLflowInferenceClient()

In [8]:
json_data = {
    "dataframe_split": {
    "columns": ["age", "default", "balance", "housing", "loan", "day", "month", "duration", "campaign", "pdays", "previous", "marital_married", "marital_single", "job_blue-collar", "job_entrepreneur", "job_housemaid", "job_management", "job_retired", "job_self-employed", "job_services", "job_student", "job_technician", "job_unemployed", "job_unknown", "education_secondary", "education_tertiary", "education_unknown", "poutcome_other", "poutcome_success", "poutcome_unknown", "contact_telephone", "contact_unknown"],
    "data": [[35, 0, 178, 0, 0, 14, 8, 76, 2, -1, 0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]]}}

In [20]:
response_json = ml_flow_online_inference_client.predict_json(json_data=json_data)
response_json.content

b'{"predictions": [0]}'

In [69]:
csv_data = """age,default,balance,housing,loan,day,month,duration,campaign,pdays,previous,marital_married,marital_single,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,education_secondary,education_tertiary,education_unknown,poutcome_other,poutcome_success,poutcome_unknown,contact_telephone,contact_unknown
35,0,178,0,0,14,8,76,2,-1,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
"""
response_csv = ml_flow_online_inference_client.predict_csv(csv_data=csv_data)
response_csv.content

b'{"predictions": [0]}'

In [43]:
pandas_data = pd.read_csv("datasets/bank-validation-dataset.csv").head(1)
print(pandas_data["y"])
pandas_data = pandas_data.drop("y", axis=1)

0    no
Name: y, dtype: object


In [44]:
response_pandas = ml_flow_online_inference_client.predict_pandas(pandas_data=pandas_data)
response_pandas.content

b'{"predictions": [0]}'

#### Batch Inference

According to [Google](https://developers.google.com/machine-learning/crash-course/production-ml-systems/static-vs-dynamic-inference), **batch inference** means the model makes predictions on a bunch of common unlabeled examples and then caches those predictions somewhere.

In [11]:
# import packages
import mlflow
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score

In [13]:
# import validation dataset
validation_df = pd.read_csv("datasets/bank-full_val.csv")
validation_df["y"] = validation_df["y"].map({"yes": 1, "no": 0})
y_validation = validation_df["y"]
validation_df = validation_df.drop("y", axis=1)

In [14]:
class ModelBatchPredictor:
    ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]
    def __init__(self):
        self.encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
        self.transformer = Transformer()

    def _apply_transformation(self, df: pd.DataFrame):
        df = self.transformer.transform(df.copy())
        encoded_df = self.encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
        df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
        df = pd.concat([df, encoded_df], axis=1)

        return df 
    
    def batch_inference_with_ml_flow(self, model_uri: str, input: pd.DataFrame):
        model = mlflow.pyfunc.load_model(model_uri)
        input = self._apply_transformation(input)
        predictions = model.predict(input)
        
        return predictions

    def batch_inference_without_ml_flow(self, model_path: str, input: pd.DataFrame):
        model = joblib.load(model_path)
        input = self._apply_transformation(input)
        predictions = model.predict(input)

        return predictions



##### Batch Inference with MLFlow

In [16]:
batch_prediction_with_mlflow = ModelBatchPredictor().batch_inference_with_ml_flow(
    model_uri='runs:/94318391fcbd491abd49234c43d4f15a/bank_model',
    input=validation_df
)

In [17]:
print(batch_prediction_with_mlflow[:5])
print("----")
print(confusion_matrix(y_validation, batch_prediction_with_mlflow))
print("----")
print(accuracy_score(y_validation, batch_prediction_with_mlflow))


[0 0 1 1 0]
----
[[3246  722]
 [ 100  454]]
----
0.8182220256523662


##### Batch Inference without MLFlow

In [18]:
batch_prediction_without_mlflow = ModelBatchPredictor().batch_inference_without_ml_flow(
    model_path=f'{PATH}lr_model.pkl',
    input=validation_df
)

In [19]:
print(batch_prediction_without_mlflow[:5])
print("----")
print(confusion_matrix(y_validation, batch_prediction_without_mlflow))
print("----")
print(accuracy_score(y_validation, batch_prediction_without_mlflow))

[0 0 1 1 0]
----
[[3246  722]
 [ 100  454]]
----
0.8182220256523662
